In [ ]:
import numpy as np
import torch
import random

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/home/duarte/Desktop/Tese/Mapping_Tese/mapping_tese")
BIDS_ROOT = PROJECT_ROOT / "data/BIDS-somatosensory/BIDS-somatosensory"
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"
EVENTS_DIR = BIDS_ROOT / "events"

RESULTS_BASE_DIR = PROJECT_ROOT / "notebooks/GNN/results/GCN_4Classes_MultiSubject"
RESULTS_BASE_DIR.mkdir(parents=True, exist_ok=True)

GRAPH_EDGES_PATH = PROJECT_ROOT / "notebooks/GNN/results/graph_edges.pt"

session = "ses-01"
task = "task-S1Map"
space = "MNI152NLin2009cAsym"
n_runs_per_subject = 4

HRF_DELAY = 6.0
WINDOW = 1

BATCH_SIZE = 16
MAX_EPOCHS = 1000
PATIENCE = 300
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.25
HIDDEN_DIM = 64

In [ ]:
import re

REGION_TO_ELECTRODES = {
    "Middle_Finger": ["E1", "E2", "E3"],
    "Hand": ["E4", "E5", "E6", "E7"],
    "Forearm": ["E8", "E9", "E10", "E11", "E12", "E13"],
    "Arm": ["E14", "E15", "E16", "E17", "E18", "E19", "E20"],
}
ELECTRODE_TO_REGION = {
    elec: region
    for region, elecs in REGION_TO_ELECTRODES.items()
    for elec in elecs
}
REGION_TO_LABEL = {"Middle_Finger": 0, "Hand": 1, "Forearm": 2, "Arm": 3}
LABEL_TO_REGION = {v: k for k, v in REGION_TO_LABEL.items()}

derivative_subjects = sorted(
    d.name for d in DERIVATIVES.iterdir()
    if d.is_dir() and d.name.startswith("sub-")
)

event_subjects = set()
for ev_path in EVENTS_DIR.glob(f"sub-*_{session}_{task}_run-*_events.tsv"):
    m = re.match(r"^(sub-[^_]+)_", ev_path.name)
    if m:
        event_subjects.add(m.group(1))
event_subjects = sorted(event_subjects)

subjects = sorted(set(derivative_subjects) & set(event_subjects))

print("Subjects in derivatives:", len(derivative_subjects))
print("Subjects in events:", len(event_subjects))
print("Subjects included:", len(subjects))
print(subjects)

In [ ]:
import json

def resolve_bold_path(subject, run, space_name):
    func_dir = DERIVATIVES / subject / session / "func"
    candidates = [
        func_dir / f"{subject}_{session}_{task}_run-{run}_space-{space_name}_desc-preproc_bold.nii.gz",
        func_dir / f"{subject}_{session}_{task}_run-{run}_space-{space_name}_desc-preproc_bold.nii",
        func_dir / f"{subject}_{session}_{task}_run-{run}_desc-preproc_bold.nii.gz",
        func_dir / f"{subject}_{session}_{task}_run-{run}_desc-preproc_bold.nii",
    ]
    for c in candidates:
        if c.exists():
            return c
    return None

def get_tr(subject, run=1, default_tr=2.0):
    func_dir = DERIVATIVES / subject / session / "func"
    json_candidates = [
        func_dir / f"{subject}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.json",
        func_dir / f"{subject}_{session}_{task}_run-{run}_desc-preproc_bold.json",
    ]
    for p in json_candidates:
        if p.exists():
            with open(p, "r", encoding="utf-8") as f:
                meta = json.load(f)
            if "RepetitionTime" in meta and meta["RepetitionTime"] is not None:
                return float(meta["RepetitionTime"])
    return float(default_tr)

tr_by_subject = {s: get_tr(s, run=1, default_tr=2.0) for s in subjects}
print("TR values:", sorted(set(tr_by_subject.values())))

In [ ]:
import pandas as pd

def load_subject_events(subject):
    all_events = []
    required_cols = {"onset", "duration", "trial_type"}

    for run in range(1, n_runs_per_subject + 1):
        ev_path = EVENTS_DIR / f"{subject}_{session}_{task}_run-{run}_events.tsv"
        if not ev_path.exists():
            raise FileNotFoundError(f"Missing events file: {ev_path}")

        df = pd.read_csv(ev_path, sep="\t")
        df.columns = [str(c).strip().lower() for c in df.columns]

        missing = required_cols - set(df.columns)
        if missing:
            raise ValueError(f"{ev_path.name} missing columns {sorted(missing)}")

        df["run"] = run
        all_events.append(df)

    events_df = pd.concat(all_events, ignore_index=True)
    stim_events = events_df[~events_df["trial_type"].isin(["Baseline", "Jitter"])].copy()
    stim_events["region"] = stim_events["trial_type"].map(ELECTRODE_TO_REGION)

    unmapped = stim_events["region"].isna().sum()
    if unmapped > 0:
        bad = sorted(stim_events.loc[stim_events["region"].isna(), "trial_type"].unique())
        raise ValueError(f"Unmapped trial_type values for {subject}: {bad}")

    stim_events["label"] = stim_events["region"].map(REGION_TO_LABEL).astype(int)
    return events_df, stim_events

_preview_e, _preview_s = load_subject_events(subjects[0])
print("Preview subject:", subjects[0])
print("Stim samples:", len(_preview_s))
print("Class counts:", _preview_s["region"].value_counts().to_dict())

In [ ]:
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.image import load_img, new_img_like

atlas = fetch_atlas_destrieux_2009()
atlas_img = load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()

s1_indices = [
    i for i, lab in enumerate(atlas.labels)
    if "L G_postcentral" in str(lab) and i != 0
]
mask_data = np.isin(atlas_data, s1_indices).astype("uint8")
s1_mask = new_img_like(atlas_img, mask_data)

print("Selected atlas indices:", len(s1_indices))

In [ ]:
def build_6nn_edge_index(voxel_coords):
    coord_to_idx = {tuple(c): i for i, c in enumerate(voxel_coords)}
    neigh = [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]

    edges = []
    for i, (x, y, z) in enumerate(voxel_coords):
        for dx, dy, dz in neigh:
            j = coord_to_idx.get((x+dx, y+dy, z+dz), None)
            if j is not None:
                edges.append([i, j])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return edge_index

In [ ]:
from nilearn.image import index_img, resample_to_img
from nilearn.maskers import NiftiMasker

first_subject = subjects[0]
first_run = resolve_bold_path(first_subject, run=1, space_name=space)
if first_run is None:
    raise FileNotFoundError(f"No first run found for {first_subject}")

first_img = load_img(str(first_run))
ref_img = index_img(first_img, 0)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation="nearest")
masker_ref = NiftiMasker(mask_img=s1_mask_resampled, standardize=None).fit(first_img)

mask_bool = masker_ref.mask_img_.get_fdata().astype(bool)
voxel_coords = np.column_stack(np.where(mask_bool))
n_voxels = voxel_coords.shape[0]

In [ ]:
if GRAPH_EDGES_PATH.exists():
    edge_index = torch.load(GRAPH_EDGES_PATH, map_location="cpu")
    print("Loaded edge_index from disk:", GRAPH_EDGES_PATH)
else:
    edge_index = build_6nn_edge_index(voxel_coords)
    torch.save(edge_index, GRAPH_EDGES_PATH)
    print("Built and saved edge_index:", GRAPH_EDGES_PATH)

In [ ]:
print("n_voxels:", n_voxels)
print("edge_index shape:", tuple(edge_index.shape))

In [ ]:
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn as nn
import torch.nn.functional as F

class SomatotopicGCN(nn.Module):
    def __init__(self, in_channels=1, hidden_channels=64, n_classes=4, dropout=0.25):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)

        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)

        self.res_proj = nn.Linear(hidden_channels, hidden_channels)
        self.classifier = nn.Linear(hidden_channels, n_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        residual = self.res_proj(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.elu(x)
        x = x + residual
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = global_mean_pool(x, batch)
        logits = self.classifier(x)
        return logits

In [ ]:
from nilearn.image import mean_img

#per subject feature extration
def extract_subject_samples(subject):
    tr_subject = tr_by_subject[subject]
    _, stim_events = load_subject_events(subject)

    run1_path = resolve_bold_path(subject, run=1, space_name=space)
    if run1_path is None:
        raise FileNotFoundError(f"Missing run-1 bold for {subject}")

    run1_img = load_img(str(run1_path))
    ref = index_img(run1_img, 0)
    local_mask = resample_to_img(s1_mask, ref, interpolation="nearest")
    masker = NiftiMasker(mask_img=local_mask, standardize=None).fit(run1_img)

    X_list, y_list, run_list = [], [], []

    for run in range(1, n_runs_per_subject + 1):
        bold_path = resolve_bold_path(subject, run=run, space_name=space)
        if bold_path is None:
            continue

        img = load_img(str(bold_path))
        run_len = img.shape[3]
        run_events = stim_events[stim_events["run"] == run].sort_values("onset")

        for _, ev in run_events.iterrows():
            center = int(np.round((float(ev["onset"]) + HRF_DELAY) / tr_subject))
            vols = list(range(max(0, center - WINDOW), min(run_len, center + WINDOW + 1)))
            if len(vols) == 0:
                continue

            avg_img = mean_img(index_img(img, vols), copy_header=True)
            feat = masker.transform(avg_img).ravel()

            if feat.shape[0] != n_voxels:
                raise ValueError(f"Voxel mismatch for {subject}, run {run}: {feat.shape[0]} vs {n_voxels}")

            X_list.append(feat.astype(np.float32))
            y_list.append(int(ev["label"]))
            run_list.append(int(run))

    if len(X_list) == 0:
        raise RuntimeError(f"No usable samples for {subject}")

    X = np.vstack(X_list)
    y = np.asarray(y_list, dtype=np.int64)
    groups = np.asarray(run_list, dtype=np.int64)

    return X, y, groups, tr_subject

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from torch_geometric.data import Data

@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    ys, ps = [], []
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.batch)
        pred = logits.argmax(dim=1).cpu().numpy()
        true = batch.y.cpu().numpy()
        ys.append(true)
        ps.append(pred)
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    acc = accuracy_score(y_true, y_pred)
    bal = balanced_accuracy_score(y_true, y_pred)
    return acc, bal, y_true, y_pred

def make_graph_dataset(X_scaled, y, edge_idx):
    data_list = []
    eidx = edge_idx.clone()
    for i in range(X_scaled.shape[0]):
        x_i = torch.tensor(X_scaled[i].reshape(-1, 1), dtype=torch.float32)
        y_i = torch.tensor(y[i], dtype=torch.long)
        data_list.append(Data(x=x_i, edge_index=eidx, y=y_i))
    return data_list

In [ ]:
import time
from sklearn.preprocessing import StandardScaler
from torch_geometric.loader import DataLoader
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import copy
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

#training cell
all_subject_rows = []

for subject in subjects:
    print("\n" + "=" * 70)
    print("Subject:", subject)
    t0 = time.time()

    subj_dir = RESULTS_BASE_DIR / subject
    fig_dir = subj_dir / "figures"
    model_dir = subj_dir / "models"
    fig_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)

    try:
        X, y, groups, tr_sub = extract_subject_samples(subject)
    except Exception as exc:
        print("Skipping subject due to extraction error:", exc)
        continue

    print("X shape:", X.shape, "y shape:", y.shape)
    print("Class counts:", dict(pd.Series(y).value_counts().sort_index()))

    fold_rows = []
    fold_cms = []
    best_overall_bal = -np.inf
    best_state = None
    best_scaler = None

    # Nested LORO: 4 outer test folds
    for test_run in range(1, n_runs_per_subject + 1):
        # Rotate validation run deterministically among remaining runs
        val_run = (test_run % n_runs_per_subject) + 1
        train_runs = [r for r in range(1, n_runs_per_subject + 1) if r != test_run and r != val_run]

        print(f"\n{subject} | Fold {test_run}/{n_runs_per_subject} | Test run {test_run} | Val run {val_run} | Train runs {train_runs}")

        train_mask = np.isin(groups, train_runs)
        val_mask = (groups == val_run)
        test_mask = (groups == test_run)

        if not np.any(train_mask) or not np.any(val_mask) or not np.any(test_mask):
            print(f"Skipping fold due to missing data split.")
            continue

        X_tr, y_tr = X[train_mask], y[train_mask]
        X_val, y_val = X[val_mask], y[val_mask]
        X_te, y_te = X[test_mask], y[test_mask]

        # Standardize features using training data parameters ONLY
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_val_s = scaler.transform(X_val)
        X_te_s = scaler.transform(X_te)

        train_graphs = make_graph_dataset(X_tr_s, y_tr, edge_index)
        val_graphs = make_graph_dataset(X_val_s, y_val, edge_index)
        test_graphs = make_graph_dataset(X_te_s, y_te, edge_index)

        train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

        # Reset model weights for each fold
        torch.manual_seed(RANDOM_SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(RANDOM_SEED)

        model = SomatotopicGCN(
            in_channels=1,
            hidden_channels=HIDDEN_DIM,
            n_classes=4,
            dropout=DROPOUT,
        ).to(device)

        # Class weights computed strictly from inner training labels
        cls = np.unique(y_tr)
        weights = compute_class_weight(class_weight="balanced", classes=cls, y=y_tr)
        full_w = np.ones(4, dtype=np.float32)
        for c, w in zip(cls, weights):
            full_w[int(c)] = w
        full_w = np.sqrt(full_w)
        full_w = full_w / full_w.mean()
        class_w_t = torch.tensor(full_w, dtype=torch.float32, device=device)

        criterion = nn.CrossEntropyLoss(weight=class_w_t)
        optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=50)

        best_fold_bal = -np.inf
        best_fold_state = None
        no_improve = 0

        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                logits = model(batch.x, batch.edge_index, batch.batch)
                loss = criterion(logits, batch.y)
                loss.backward()
                optimizer.step()

            # Evaluate on VALIDATION split (never on test set)
            val_acc, val_bal, _, _ = evaluate_model(model, val_loader, device)
            scheduler.step(val_bal)

            if val_bal > best_fold_bal:
                best_fold_bal = val_bal
                best_fold_state = copy.deepcopy(model.state_dict())
                no_improve = 0
            else:
                no_improve += 1

            if no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch} | best val bal acc = {best_fold_bal*100:.2f}%")
                break

        if best_fold_state is None:
            print(f"Fold with test run {test_run} failed for {subject}")
            continue

        # Final evaluation on outer held-out TEST run (touched exactly once)
        model.load_state_dict(best_fold_state)
        te_acc, te_bal, y_true, y_pred = evaluate_model(model, test_loader, device)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])

        fold_rows.append({
            "subject": subject,
            "test_run": test_run,
            "val_run": val_run,
            "train_runs": train_runs,
            "accuracy": float(te_acc),
            "balanced_accuracy": float(te_bal),
        })
        fold_cms.append(cm)

        print(f"Test run {test_run} | Val run {val_run} -> Test Acc={te_acc*100:.2f}% | Test Bal Acc={te_bal*100:.2f}%")

        if te_bal > best_overall_bal:
            best_overall_bal = te_bal
            best_state = copy.deepcopy(best_fold_state)
            best_scaler = copy.deepcopy(scaler)

    if len(fold_rows) == 0:
        print("No valid folds for subject:", subject)
        continue

    fold_df = pd.DataFrame(fold_rows).sort_values("fold").reset_index(drop=True)
    fold_df.to_csv(subj_dir / "fold_balanced_accuracy.csv", index=False)

    cm_total = np.sum(np.stack(fold_cms, axis=0), axis=0)
    mean_acc = float(fold_df["accuracy"].mean())
    std_acc = float(fold_df["accuracy"].std(ddof=0))
    mean_bal = float(fold_df["balanced_accuracy"].mean())
    std_bal = float(fold_df["balanced_accuracy"].std(ddof=0))

    y_all_true = []
    y_all_pred = []
    for cm in fold_cms:
        pass

    # Recover per-class metrics from fold-level predictions is expensive to cache here.
    # Approximate from cm_total for recall; precision/f1 via sklearn from reconstructed vectors is skipped.
    per_class_recall = np.diag(cm_total) / np.maximum(cm_total.sum(axis=1), 1)

    torch.save(best_state, model_dir / "best_gcn_model.pt")
    with open(model_dir / "best_scaler.pkl", "wb") as f:
        pickle.dump(best_scaler, f)
    np.save(model_dir / "confusion_matrix_total.npy", cm_total)

    # confusion matrix figure
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm_total,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[LABEL_TO_REGION[i] for i in range(4)],
        yticklabels=[LABEL_TO_REGION[i] for i in range(4)],
        ax=ax,
    )
    ax.set_title(f"{subject} | bal={mean_bal*100:.1f}%")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.tight_layout()
    plt.savefig(fig_dir / "confusion_matrix_gcn.png", dpi=150)
    plt.close()

    all_subject_rows.append({
        "subject": subject,
        "mean_accuracy": mean_acc,
        "std_accuracy": std_acc,
        "mean_balanced_accuracy": mean_bal,
        "std_balanced_accuracy": std_bal,
        "macro_recall": float(np.mean(per_class_recall)),
        "per_class_recall": str(per_class_recall.tolist()),
        "n_samples_subject": int(X.shape[0]),
        "n_features_subject": int(X.shape[1]),
        "tr_subject": float(tr_sub),
    })

    print(f"Subject done in {(time.time()-t0):.1f}s | mean bal={mean_bal*100:.2f}%")

In [ ]:
if len(all_subject_rows) == 0:
    raise RuntimeError("No subject produced valid GCN results.")

results_df = pd.DataFrame(all_subject_rows).sort_values("subject").reset_index(drop=True)
results_csv = RESULTS_BASE_DIR / "all_subjects_results_gcn.csv"
results_df.to_csv(results_csv, index=False)

print(results_df[["subject", "mean_balanced_accuracy", "std_balanced_accuracy"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(max(7, len(results_df)*1.0), 4))
x = np.arange(len(results_df))
ax.bar(
    x,
    results_df["mean_balanced_accuracy"] * 100,
    yerr=results_df["std_balanced_accuracy"] * 100,
    capsize=5,
    color="#3b82f6",
    ecolor="black",
)
ax.axhline(25, color="red", linestyle="--", label="Chance (25%)")
ax.set_xticks(x)
ax.set_xticklabels(results_df["subject"], rotation=45, ha="right")
ax.set_ylabel("Balanced Accuracy (%)")
ax.set_title("GCN Multi-subject LORO")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_BASE_DIR / "group_balanced_accuracy_gcn.png", dpi=150)
plt.show()

print("Saved:", results_csv)